In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")

In [0]:
from pyspark.sql import functions as F

 

# Read approval from the persistent table.
# This notebook does not depend on earlier Python variables.
control_rows = (
    spark.table(f"{table_prefix}.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(control_rows) != 1:
    raise RuntimeError("Expected exactly one batch control record")

control = control_rows[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved; Silver load blocked")

metadata = (
    spark.table(f"{table_prefix}.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
)

inventory = (
    spark.table(f"{table_prefix}.bronze_file_inventory")
    .filter(F.col("batch_id") == batch_id)
)

silver = (
    metadata.alias("m")
    .join(inventory.alias("i"), ["batch_id", "file_name"], "inner")
    .select(
        F.col("batch_id"),
        F.trim(F.col("m.document_id")).alias("document_id"),
        F.trim(F.col("m.account_number")).alias("account_number"),
        F.trim(F.col("m.customer_name")).alias("customer_name"),
        F.upper(F.trim(F.col("m.document_type"))).alias("document_type"),
        F.expr("try_cast(m.statement_date AS DATE)").alias("statement_date"),
        F.col("file_name"),
        F.col("i.pdf_path"),
        F.lower(F.trim(F.col("m.source_sha256"))).alias("source_sha256"),
        F.col("i.file_size_bytes"),
        F.lit(control["validation_run_id"]).alias("validation_run_id"),
        F.current_timestamp().alias("processed_at"),
    )
)

# Guard against lost rows and duplicate keys after transformation.
expected_count = control["expected_metadata_count"]

if silver.count() != expected_count:
    raise RuntimeError("Silver row count differs from the approved batch")

for key in ["document_id", "file_name"]:
    duplicates = silver.groupBy(key).count().filter(F.col("count") > 1)
    if duplicates.limit(1).count() > 0:
        raise RuntimeError(f"Duplicate {key}; Silver load blocked")

if silver.filter(F.col("statement_date").isNull()).limit(1).count() > 0:
    raise RuntimeError("Invalid statement date; Silver load blocked")

silver.createOrReplaceTempView("current_silver_batch")

spark.sql(f"""
    MERGE INTO {table_prefix}.silver_documents AS target
    USING current_silver_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN MATCHED
         AND target.validation_run_id <> source.validation_run_id
         THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table(f"{table_prefix}.silver_documents")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Silver documents: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT document_id) AS unique_documents,
    COUNT(DISTINCT validation_run_id) AS validation_runs,
    COUNT_IF(statement_date IS NULL) AS invalid_dates
FROM IDENTIFIER(:catalog || '.' || :schema || '.silver_documents')
WHERE batch_id = :batch_id;